In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv("../data/superstore.csv", encoding="latin1")
df.head()

In [ ]:
df.info()

df.describe(include="all")

In [ ]:
df = df.dropna().copy()

df["Order Date"] = pd.to_datetime(df["Order Date"])

# sort for time consistency
df = df.sort_values("Order Date")

In [ ]:
monthly_sales = df.groupby(pd.Grouper(key="Order Date", freq="M"))["Sales"].sum()

plt.figure(figsize=(14,6))
plt.plot(monthly_sales, marker="o")
plt.title("📈 Monthly Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(True)
plt.savefig("../outputs/monthly_sales_trend.png")
plt.show()

In [ ]:
sales_df = df.groupby("Order Date")["Sales"].sum().reset_index()
sales_df = sales_df.sort_values("Order Date")

# time features
sales_df["Year"] = sales_df["Order Date"].dt.year
sales_df["Month"] = sales_df["Order Date"].dt.month
sales_df["Day"] = sales_df["Order Date"].dt.day

# lag feature (captures trend)
sales_df["Lag_1"] = sales_df["Sales"].shift(1)

# remove missing values created by shift
sales_df = sales_df.dropna()

In [ ]:
X = sales_df[["Year", "Month", "Day", "Lag_1"]]
y = sales_df["Sales"]

In [ ]:
split = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("📊 Model Performance")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(y_test.values, label="Actual", linewidth=2)
plt.plot(pred, label="Predicted", linewidth=2)

plt.title("📉 Actual vs Predicted Sales")
plt.xlabel("Time")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)
plt.savefig("../outputs/actual_vs_predicted.png")
plt.show()

In [ ]:
future_dates = pd.date_range(
    start=sales_df["Order Date"].max() + pd.Timedelta(days=1),
    periods=30
)

# use last known lag value
last_lag = sales_df["Sales"].iloc[-1]

future_df = pd.DataFrame({
    "Year": future_dates.year,
    "Month": future_dates.month,
    "Day": future_dates.day,
    "Lag_1": last_lag
})

future_pred = model.predict(future_df)

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(future_dates, future_pred, color="green", marker="o")

plt.title("📈 30-Day Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Predicted Sales")
plt.grid(True)

plt.xticks(rotation=45)
plt.savefig("../outputs/future_forecast.png")
plt.show()